# UC2 — Metal Surface (Magnetic Tile): 5. Pseudo-Labeling

Turn the synthetic Metal Surface images into a **fully annotated dataset**: refined
masks, COCO-format bounding boxes + instance masks, class folders, and (optionally)
natural-language captions — ready for downstream detection / segmentation / VLM training.

Pipeline: **mask clustering (DBSCAN) → optional SAM2 mask refinement → bbox & RLE →
optional Qwen3-VL captioning → organize outputs**.

> **How commands run in this tutorial.** All pipeline steps run inside the
> `cosmos-predict2` conda environment. In a notebook cell we prefix shell
> commands with `conda run -n cosmos-predict2` (add `--live-stream` to stream
> logs live). If you prefer, open a JupyterLab **Terminal**, run
> `conda activate cosmos-predict2` once, and paste the same commands without
> the `conda run` prefix.
>
> If that environment does not exist yet, build it first with the top-level
> [tutorial/notebooks/0-setup-cuda128.ipynb](../../0-setup-cuda128.ipynb) — see
> the prerequisite note below.

## 5.0 Set the project root

In [ ]:
# Resolve the repository root (the folder containing pyproject.toml) and cd into it,
# so every relative path below (datasets/, checkpoints/, results/, scripts/) resolves.
import os
d = os.getcwd()
while d != "/" and not os.path.exists(os.path.join(d, "pyproject.toml")):
    d = os.path.dirname(d)
LOCAL_PROJECT_DIR = d
os.chdir(LOCAL_PROJECT_DIR)
# Pipeline scripts read the finetuned models & write outputs under the repo root.
os.environ.setdefault("IMAGINAIRE_OUTPUT_ROOT", "./results")
print("Project root:", LOCAL_PROJECT_DIR)

## 5.1 (Optional) Fine-tune SAM2

SAM2 refines the generated masks. The pretrained `sam2.1_hiera_large.pt` works out of
the box; fine-tuning on your defects sharpens boundaries. Combine all defects into one
flat directory, then (from a terminal, GPU-heavy):

```bash
conda run -n cosmos-predict2 python -m scripts.anomaly_gen.finetune_sam \
    --image_dir=<combined>/anomaly_image \
    --mask_dir=<combined>/mask \
    --output_dir=results/UC2_metal/sam2_finetuning \
    --epochs=20
```

To skip refinement entirely, pass `--no_mask_refinement` to the pseudo-labeling step below.

## 5.2 Run pseudo-labeling

Runs on the generation output from notebook 4. `--no_caption` skips the (slow)
Qwen3-VL captioner — drop it to also generate captions.

In [ ]:
!conda run -n cosmos-predict2 python -m scripts.anomaly_gen.pseudo_label \
    --ori_image_dir=results/UC2_metal/example_output/original_image \
    --gen_image_dir=results/UC2_metal/example_output/reconstructed_image \
    --mask_dir=results/UC2_metal/example_output/original_mask \
    --csv_path=results/UC2_metal/example_output/SDG_result.csv \
    --output_dir=results/UC2_metal/pseudo_labeling \
    --no_caption

## 5.3 Output structure

```
results/UC2_metal/pseudo_labeling/
├── classification/           # images organized by class (+ classes.txt) for TAO
├── images/                   # generated anomaly images
├── masks/                    # refined binary masks
├── visualization/            # images with mask + bbox overlaid
├── coco_annotations.json     # COCO bboxes + instance masks (after refinement)
└── ori_coco_annotations.json # COCO annotations before refinement
```

#### Example annotations

The visualization overlays the refined mask + bounding box on each generated image.

| Defect | Annotated (mask + bbox) |
|---|---|
| metal_surface+MT_Blowhole | ![](assets/pseudo_label/metal_surface+MT_Blowhole_viz.png) |
| metal_surface+MT_Break | ![](assets/pseudo_label/metal_surface+MT_Break_viz.png) |
| metal_surface+MT_Crack | ![](assets/pseudo_label/metal_surface+MT_Crack_viz.png) |
| metal_surface+MT_Fray | ![](assets/pseudo_label/metal_surface+MT_Fray_viz.png) |
| metal_surface+MT_Uneven | ![](assets/pseudo_label/metal_surface+MT_Uneven_viz.png) |


## Workflow Complete

You have run the full Cosmos AnomalyGen pipeline for **UC2: Metal Surface (Magnetic Tile)**:

1. **Setup** — environment, Hugging Face auth, checkpoints.
2. **Dataset preparation** — fetched & organized the Metal Surface data.
3. **Training** — fine-tuned the adapters (or used the released checkpoint).
4. **Auto Mask Placement** — built the generation testcase (`free` routing).
5. **Generation** — synthesized anomaly images and evaluated them.
6. **Pseudo-Labeling** — produced COCO annotations & class folders.

The annotated synthetic dataset lives at `results/UC2_metal/pseudo_labeling/` — use it
directly as training data for downstream anomaly detection, instance segmentation, or
vision-language models.

**Prefer one command?** [6-agentic-flow](./6-agentic-flow.ipynb) runs steps 2→5 from a
single Claude Code prompt, with an automatic per-sample quality search on top.